In [53]:
# get the UserID of a channel
import os
from dotenv import load_dotenv
load_dotenv()
import requests


# https://twitchtokengenerator.com/
TOKEN = os.environ["ACCESS_TOKEN"]
CLIENT_ID = os.environ["CLIENT_ID"]

def getUserID(username):
    headers = {
        "Authorization": f"Bearer {TOKEN}",
        "Client-Id": CLIENT_ID
    }

    r = requests.get(
        f"https://api.twitch.tv/helix/users?login={username}",
        headers=headers
    )

    data = r.json()["data"]

    if not data:
        return None

    return data[0]["id"]

target_channel = "kumi"
target_UserID = getUserID(target_channel)
print(f"channel: {target_channel}, UserID: {target_UserID}")

channel: kumi, UserID: 142260777


In [55]:
# use the user id, to get 7tv emotes
import requests
import os
from dotenv import load_dotenv
load_dotenv()

def get_7tv_emote_set(user_id):
    url = f"https://7tv.io/v3/users/twitch/{user_id}"
    data = requests.get(url).json()
    emote_set = {}

    if "emote_set" in data:
        emotes = data["emote_set"]["emotes"]
        for emote in emotes:
            emote_name = emote["name"]
            emote_id = emote["id"]
            emote_url = f"https://cdn.7tv.app/emote/{emote_id}/4x.webp"
            emote_set[emote_name] = emote_url
    else:
        print("No 7TV emote set found")

    return emote_set

_7tv_emote_set = get_7tv_emote_set(target_UserID)
_7tv_emote_set

{'Skrunkle': 'https://cdn.7tv.app/emote/01H5NBGZCR0004J7X9EEXWJ9J6/4x.webp',
 'SNIFFA': 'https://cdn.7tv.app/emote/01F7M225F8000AWSXNQ65M4PKG/4x.webp',
 'ratJAM': 'https://cdn.7tv.app/emote/01F6QV6G8R0000TEKRM6BFG0Z3/4x.webp',
 'catJAM': 'https://cdn.7tv.app/emote/01F6MQ33FG000FFJ97ZB8MWV52/4x.webp',
 'LICKA': 'https://cdn.7tv.app/emote/01G78WXYZ00003P60HPZKBCA6X/4x.webp',
 'chipichipi': 'https://cdn.7tv.app/emote/01HHDNPXMG00019YQXMRBV26JR/4x.webp',
 'wtf': 'https://cdn.7tv.app/emote/01G1N8W84800087QC0373CF714/4x.webp',
 'oh': 'https://cdn.7tv.app/emote/01HQ5S4MW80008FKJ26EWGDFW9/4x.webp',
 'KumiFlashbang': 'https://cdn.7tv.app/emote/01HYRVWMAR0000HRSXF2T6TWSF/4x.webp',
 'YIPPEE': 'https://cdn.7tv.app/emote/01H04ZFGF00006G2W394DN8QWG/4x.webp',
 'CAUGHT': 'https://cdn.7tv.app/emote/01H0SQNM9R0005HNCSM10SYJEQ/4x.webp',
 'veryCat': 'https://cdn.7tv.app/emote/01GMAH9MB000066S7TTNVB1TGD/4x.webp',
 'KEYS': 'https://cdn.7tv.app/emote/01GBRJBJMG000BS4BKZQQDARSQ/4x.webp',
 'glorp': 'https://cd

In [56]:
# use the user id, to get BetterTwitchTV emotes
def get_BTV_emote_set(user_id):
    url = f"https://api.betterttv.net/3/cached/users/twitch/{user_id}"
    data = requests.get(url).json()
    emote_set = {}

    if "sharedEmotes" in data:
        emotes = data["sharedEmotes"]

        for emote in emotes:
            emote_name = emote["code"]
            emote_id = emote["id"]
            emote_url = f"https://cdn.betterttv.net/emote/{emote_id}/3x.webp"
            emote_set[emote_name] = emote_url
    else:
        print("No BTV emote set found")

    return emote_set

BTV_emote_set = get_BTV_emote_set(142260777)
BTV_emote_set

{'pepeDS': 'https://cdn.betterttv.net/emote/5b444de56b9160327d12534a/3x.webp',
 'PepoCheer': 'https://cdn.betterttv.net/emote/5abd36396723dc149c678e90/3x.webp',
 'RainbowPls': 'https://cdn.betterttv.net/emote/5b35cae2f3a33e2b6f0058ef/3x.webp',
 'POGGERS': 'https://cdn.betterttv.net/emote/58ae8407ff7b7276f8e594f2/3x.webp',
 'pugPls': 'https://cdn.betterttv.net/emote/5de88ccef6e95977b50e6eb1/3x.webp',
 'KLEEMAD': 'https://cdn.betterttv.net/emote/602d59bdee839b1e5ec6dad4/3x.webp',
 'SNIFFA': 'https://cdn.betterttv.net/emote/60be7fa7f8b3f62601c3a4b2/3x.webp'}

In [64]:
# create the paths if not made already
# create the structure
# channel name/
# * _7tv
# * bttv
# * ffz
from pathlib import Path
base_dir = Path(f"emote_cache/{target_channel}")

for folder in ["_7tv", "bttv", "ffz"]:
    (base_dir / folder).mkdir(parents=True, exist_ok=True)

In [61]:
# download and save the files
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor

session = requests.Session()

def download(item, output_dir):
    name, url = item

    try:
        r = session.get(url, timeout=10)
        r.raise_for_status()

        ext = os.path.splitext(urlparse(url).path)[1] or ".png"

        with open(output_dir / f"{name}{ext}", "wb") as f:
            f.write(r.content)

        return f"✓ {name}"
    except Exception as e:
        return f"✗ {name}: {e}"

In [63]:
# use threads to download the urls from 7tv set
from functools import partial

output_dir = Path(f"emote_cache/{target_channel}/_7tv")
output_dir.mkdir(parents=True, exist_ok=True)

_7tv_emote_set = get_7tv_emote_set(target_UserID)

with ThreadPoolExecutor(max_workers=20) as executor:
    for result in executor.map(
        partial(download, output_dir=output_dir),
        _7tv_emote_set.items()
    ):
        print(result)

✓ Skrunkle
✓ SNIFFA
✓ ratJAM
✓ catJAM
✓ LICKA
✓ chipichipi
✓ wtf
✓ oh
✓ KumiFlashbang
✓ YIPPEE
✓ CAUGHT
✓ veryCat
✓ KEYS
✓ glorp
✓ kumiSteer
✓ NutDog
✓ hiii
✓ SCATTER
✓ KEKrunes
✓ catErm
✓ WideAmongUsGaySex
✓ !play
✓ cinema
✓ ADHD
✓ LMAO
✓ Adge
✓ erm
✓ Sadge
✓ mhm
✓ catArrive
✓ ICANT
✓ ok
✓ PauseChamp
✓ AINTNOWAY
✓ EWWW
✓ LUBBERS
✓ modCheck
✓ MONKE
✓ Waiting
✓ SoCute
✓ imNOTcrying
✓ SUSSY
✓ NODDERS
✓ NOWAYING
✓ catsittingverycomfortable
✓ Devious
✓ Flushed
✓ INSANECAT
✓ juh
✓ peepoWow
✓ peepoShy
✓ PETTHEMODS
✓ bye
✓ CatDance
✓ LifeTogether
✓ o7
✓ modskillthatguy
✓ FeelsLagMan
✓ GLAUGHT
✓ Timeout
✓ hiFirstTimeChatter
✓ ASSEMBLE
✓ stopbeingMean
✓ catMunch
✓ HYPERS
✓ Life
✓ peepoWeirdlooking
✓ PogFish
✓ THEVOICES
✓ happie
✓ NOHORNY
✓ ItsglorpingTime
✓ :3
✓ Oldge
✓ clappi
✓ popCat
✓ WHAT
✓ uia
✓ NOOO
✓ wiwiwi
✓ GoodTake
✓ BangerBand
✓ Jigglin
✓ WIDE
✓ WHERE
✓ stupid
✓ plegh
✓ RAW
✓ glinton
✓ kumiPlotting
✓ SMILERS
✓ WaitingRIOT
✓ LETSFUCKINGPISS
✓ jah
✓ AREYOUAGIRL
✓ Smadge
✓ WICKED
✓ MADG

In [65]:
# use threads to download the urls from bttv set
from functools import partial

output_dir = Path(f"emote_cache/{target_channel}/bttv")
output_dir.mkdir(parents=True, exist_ok=True)

btv_emote_set = get_BTV_emote_set(target_UserID)

with ThreadPoolExecutor(max_workers=20) as executor:
    for result in executor.map(
        partial(download, output_dir=output_dir),
        btv_emote_set.items()
    ):
        print(result)

✓ pepeDS
✓ PepoCheer
✓ RainbowPls
✓ POGGERS
✓ pugPls
✓ KLEEMAD
✓ SNIFFA
